In [3]:
import pandas as pd
import numpy as np

def compute_baseline_threshold_from_log(log_path, target_success_rate=0.25):
    """
    Computes threshold so that about `target_success_rate` of trials
    would have succeeded during baseline.

    Uses per-trial maximum relAvgDff over active trial periods only.
    Assumes the log file has columns including:
        - trial   (0 during rest, positive integer during trial)
        - rew_threshold
        - reward
        - relAvgDff column (name depends on ROI operation)
    """
    df = pd.read_csv(log_path, sep='\t')

    # Find the combined dff column automatically:
    candidate_cols = [c for c in df.columns if c not in
                      ['frame', 'time', 'freq', 'rew_threshold', 'reward', 'trial', 'audio', 'tot_rewards']]

    # Usually the combined ROI-operation column is the 3rd dff-like one.
    # Safer: choose the last numeric signal column before freq if possible.
    numeric_candidates = []
    for c in candidate_cols:
        try:
            pd.to_numeric(df[c], errors='raise')
            numeric_candidates.append(c)
        except Exception:
            pass

    if len(numeric_candidates) == 0:
        raise ValueError("Could not identify relAvgDff column automatically.")

    # In your current log format, the combined relAvgDff is typically the last of the dff signal columns
    rel_col = numeric_candidates[-1]

    # Keep active trial frames only
    df_trials = df[df['trial'] > 0].copy()
    if df_trials.empty:
        raise ValueError("No active trial frames found in log.")

    # Per-trial peak value
    trial_peaks = df_trials.groupby('trial')[rel_col].max()

    # Threshold so top 25% of trial peaks would count as success
    q = 1.0 - target_success_rate
    threshold = float(trial_peaks.quantile(q))

    # Estimate achieved success rate with that threshold
    estimated_success_rate = float((trial_peaks > threshold).mean())

    out = {
        'threshold': threshold,
        'target_success_rate': target_success_rate,
        'estimated_success_rate': estimated_success_rate,
        'n_trials': int(trial_peaks.shape[0]),
        'relAvgDff_column': rel_col,
        'trial_peaks': trial_peaks
    }
    return out

In [2]:
# EXAMPLE USE
result = compute_baseline_threshold_from_log(
    log_path='/home/jet/sentech_2roi/567312l3/20260202152658/S1/567312l3_20260202152658.txt',
    target_success_rate=0.25
)

print("Threshold:", result['threshold'])
print("Estimated success rate:", result['estimated_success_rate'])
print("N trials:", result['n_trials'])
print("Using signal column:", result['relAvgDff_column'])

Threshold: 230.8833725
Estimated success rate: 0.25
N trials: 100
Using signal column: TR_LnoiseSTD


In [ ]:
# EXAMPLE USE
result = compute_baseline_threshold_from_log(
    log_path='path/to/your_baseline_log.txt',
    target_success_rate=0.25
)

print("Threshold:", result['threshold'])
print("Estimated success rate:", result['estimated_success_rate'])
print("N trials:", result['n_trials'])
print("Using signal column:", result['relAvgDff_column'])

In [5]:
import pandas as pd

def compute_threshold_from_trial_summary(csv_path, target_success_rate=0.25):
    df = pd.read_csv(csv_path)
    threshold = float(df['peak_relAvgDff'].quantile(1.0 - target_success_rate))
    estimated = float((df['peak_relAvgDff'] > threshold).mean())
    return threshold, estimated, len(df)

In [50]:
import os

data_path = r"/home/jet/sentech_2roi/"
mouse = "573102m3_base"
day = "20260424162234"
csv_path = data_path + mouse + os.sep + day + os.sep + "S1" + os.sep + f'{mouse}_{day}_baseline_trial_summary.csv'

threshold, estimated, length = compute_threshold_from_trial_summary(csv_path, 0.35)
print(threshold)
print(estimated)
print(length)

0.039706275901015844
0.36
25
